# pilot_analysis.ipynb — Pilot Experiment Analysis (N=80)
**Nhóm 3 — SWT302** | Mục đích: kiểm tra pipeline trước khi scale full run

> Chạy sau khi đã có `pilot_llm_output.csv`. Không cần metric script đầy đủ — chỉ kiểm tra format và phân phối sơ bộ.

In [ ]:
# Cell 1 — Load pilot output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

PILOT_PATH = 'pilot_llm_output.csv'
df = pd.read_csv(PILOT_PATH)
print(f'Pilot samples: {len(df)}')
print(f'Columns: {list(df.columns)}')
print(f'Null values: {df.isnull().sum().sum()}')
print()
print(df.head(2)[['sample_id','ground_truth']].to_string())

## 1. Kiểm tra format output (4 section)

In [ ]:
# Cell 2 — Kiểm tra Qwen output có đủ 4 section không
REQUIRED_SECTIONS = [
    r'Steps to Reproduce|S2R|steps to reproduce',
    r'Expected Result|Expected Behavior|ER',
    r'Actual Result|Actual Behavior|AR',
    r'Additional Info|Additional Information',
]

def check_sections(text):
    return all(re.search(p, text, re.IGNORECASE) for p in REQUIRED_SECTIONS)

df['has_all_sections'] = df['qwen_output'].astype(str).apply(check_sections)
valid = df['has_all_sections'].sum()
print(f'Có đủ 4 section: {valid}/{len(df)} ({valid/len(df)*100:.1f}%)')
print(f'Thiếu section:   {len(df)-valid}/{len(df)} ({(len(df)-valid)/len(df)*100:.1f}%)')
print()
if valid < len(df):
    bad = df[~df['has_all_sections']][['sample_id']]
    print(f'Sample IDs thiếu section: {bad["sample_id"].tolist()}')
else:
    print('✅ 100% samples có đủ 4 section — pipeline hoạt động đúng')

## 2. Phân phối độ dài output

In [ ]:
# Cell 3 — So sánh độ dài ground truth vs Qwen output
df['gt_len']   = df['ground_truth'].astype(str).str.split().str.len()
df['qwen_len'] = df['qwen_output'].astype(str).str.split().str.len()
ratio = (df['qwen_len'] / df['gt_len']).mean()

print('Ground Truth — độ dài (từ):')
print(df['gt_len'].describe().round(1).to_string())
print()
print('Qwen Output — độ dài (từ):')
print(df['qwen_len'].describe().round(1).to_string())
print(f'\nTỷ lệ Qwen/GT (trung bình): {ratio:.2f}x')
if ratio > 1.5:
    print('⚠️  Qwen viết dài hơn GT đáng kể — có thể ảnh hưởng SBERT')

# Plot
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(df['gt_len'],   bins=20, alpha=0.6, label='Ground Truth', color='steelblue')
ax.hist(df['qwen_len'], bins=20, alpha=0.6, label='Qwen Output',  color='coral')
ax.set_xlabel('Số từ'); ax.set_ylabel('Count')
ax.set_title(f'Phân phối độ dài văn bản — Pilot (N=80)\nTỷ lệ Qwen/GT = {ratio:.2f}x')
ax.legend()
plt.tight_layout(); plt.savefig('pilot_length_distribution.png', dpi=150); plt.show()

## 3. Nhận xét pilot & Quyết định tiếp theo

In [ ]:
# Cell 4 — Tóm tắt quyết định
print('=== PILOT ANALYSIS SUMMARY ===')
print(f'N pilot: {len(df)}')
print(f'Valid output (4 sections): {df["has_all_sections"].sum()} ({df["has_all_sections"].mean()*100:.1f}%)')
print(f'Qwen output dài hơn GT: {ratio:.2f}x trung bình')
print()
print('Quyết định sau pilot:')
print('  ✅ Pipeline chạy đúng, format output đúng → tiến hành full run (N=262)')
print('  ✅ Không có empty response (0/80)')
print('  ✅ Không cần amendment')
print('  ⚠️  Ghi nhận: Qwen output dài hơn GT ~1.77x (pilot); ~1.82x trên full run N=262) → có thể ảnh hưởng SBERT')
print('      → Ghi vào notes.md, KHÔNG thay đổi metric/threshold')